In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()

In [2]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('MIRROR_DB_SERVER')};"
    "DATABASE=PROVENZAL;"
    f"UID={os.getenv('MIRROR_DB_USER')};"
    f"PWD={os.getenv('MIRROR_DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
fecha_corte = pd.to_datetime("2025-11-30")

# Rango de análisis
fecha_inicio_12m = fecha_corte - pd.DateOffset(months=12)
fecha_inicio_24m = fecha_corte - pd.DateOffset(months=24)

In [4]:
# ===============================
# Datos de clientes y ventas
# ===============================
df_clientes = pd.read_sql(f""" 
SELECT Cliente, MIN(Fecha) AS CosechaFecha
FROM Ventas_CRM
WHERE Canal IN ('Chat Center','Retail','WhatsApp','Ecommerce','Ferias')
GROUP BY Cliente
HAVING MIN(Fecha) <= '{fecha_corte}'""", engine)
df_clientes["CosechaFecha"] = pd.to_datetime(df_clientes["CosechaFecha"], errors="coerce")

query_ventas = f"""
SELECT Cliente, Fecha, Venta_Neta AS Venta
FROM Ventas_CRM
WHERE Fecha BETWEEN '{fecha_inicio_24m:%Y-%m-%d}' AND '{fecha_corte:%Y-%m-%d}'
    AND Canal IN ('Chat Center','Retail','WhatsApp','Ecommerce','Ferias')
"""
df_ventas = pd.read_sql(query_ventas, engine)
df_ventas["Fecha"] = pd.to_datetime(df_ventas["Fecha"], errors="coerce")

In [5]:
# ===============================
# 3) Métricas por cliente
# ===============================
# Última compra 
ultima_compra = (
    df_ventas.groupby("Cliente")["Fecha"]
    .max()
    .reset_index()
    .rename(columns={"Fecha": "UltimaCompra"})
)

# Ventas últimos 12 meses (para SegmentoActual)
ventas_12m = (
    df_ventas[df_ventas["Fecha"] >= fecha_inicio_12m]
    .groupby("Cliente")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta12M"})
)

# Ventas 24 meses (para Segmento24M)
ventas_24m = (
    df_ventas.groupby("Cliente")["Venta"]
    .sum()
    .reset_index()
    .rename(columns={"Venta": "Venta24M"})
)

# ===============================
# 4) Merge sobre la base completa de clientes
# ===============================
clientes = df_clientes.merge(ultima_compra, on="Cliente", how="left")
clientes = clientes.merge(ventas_12m, on="Cliente", how="left")
clientes = clientes.merge(ventas_24m, on="Cliente", how="left")


In [6]:
# ===============================
# 5) Función exacta para asignar segmento por monto 
# ===============================
def segmento_por_valor(valor):
    # Nota: 0 => "Sin Segmento"
    if pd.isna(valor) or valor == 0:
        return "Sin Segmento"
    if valor > 1_400_000:
        return "Diamante"
    if 700_000 <= valor <= 1_400_000:
        return "Oro"
    if 300_000 <= valor < 700_000:
        return "Plata"
    if 0 < valor < 300_000:
        return "Bronce"
    return "Sin Segmento"

clientes["Segmento12M"] = clientes["Venta12M"].apply(segmento_por_valor)
clientes["Segmento24M"] = clientes["Venta24M"].apply(segmento_por_valor)

# ===============================
# 6) Recencia en días 
# ===============================
# calcular dias desde ultima compra 
clientes["RecenciaDias"] = (fecha_corte - clientes["UltimaCompra"]).dt.days

def calcular_recencia(row):
    # Si no tiene ultima compra -> devolvemos 
    if pd.isna(row["UltimaCompra"]):
        return None

    # Nuevo: basado en cosecha (<= 90 dias)
    if pd.notna(row["CosechaFecha"]):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= 90:
            return "Nuevo"

    d = row["RecenciaDias"]
    if d <= 120:
        return "Muy Activo"
    if 121 <= d <= 300:
        return "Activo"
    if 301 <= d <= 365:
        return "Por Inactivar"
    if 366 <= d <= 395:
        return "Churn"
    if d > 395:
        return "Inactivo"
    return None

clientes["Recencia"] = clientes.apply(calcular_recencia, axis=1)

# ===============================
# 7) Definir Segmento final 
# ===============================
def asignar_segmento_final(row):
    rec = row["Recencia"]
    if rec in ["Nuevo", "Muy Activo", "Activo", "Por Inactivar"]:
        return row["Segmento12M"]
    if rec in ["Churn", "Inactivo"]:
        return row["Segmento24M"]
    # Si no tiene recencia (no compró) -> si tiene ventas24M ==0 -> Sin Segmento
    return "Sin Segmento"

clientes["Segmento"] = clientes.apply(asignar_segmento_final, axis=1)

# ===============================
# 8) Regla final: si Segmento 
# ===============================
clientes.loc[clientes["Segmento"] == "Sin Segmento", "Recencia"] = None

In [7]:
# ===============================
# 9) Resultado final (columns)
# ===============================
resultado = clientes[[
    "Cliente", "CosechaFecha", "UltimaCompra",
    "Venta12M", "Venta24M",
    "Recencia", "Segmento"
]]

# Mostrar ejemplo
print(resultado.head(50))

          Cliente CosechaFecha UltimaCompra   Venta12M    Venta24M  \
0        28296992   2019-11-14          NaT        NaN         NaN   
1    C 1019115539   2024-01-20   2024-01-20        NaN   243697.49   
2     C 102546113   2023-05-16          NaT        NaN         NaN   
3     C 109872559   2022-11-30          NaT        NaN         NaN   
4      C 14940190   2020-06-17          NaT        NaN         NaN   
5      C 20087646   2020-02-29          NaT        NaN         NaN   
6     C 201610320   2022-01-07          NaT        NaN         NaN   
7      C 22429607   2019-12-19          NaT        NaN         NaN   
8      C 28811945   2025-05-21   2025-05-21  834700.00   834700.00   
9      C 32205379   2023-10-27          NaT        NaN         NaN   
10     C 34550936   2024-06-12   2024-06-12        NaN   789650.00   
11     C 42076183   2024-07-19   2024-07-19        NaN   205500.00   
12     C 51670219   2024-04-20   2024-04-20        NaN   174000.00   
13     C 52255465   

In [8]:
resultado.to_excel("segmentacion_clientes2.xlsx", index=False)

In [9]:
resultado.to_sql("Segmentacion_Clientes", engine, if_exists="replace", index=False)

272